In [69]:
import os
import re
import contractions

In [70]:
def extract_subtitles_with_timestamps(ass_file, output_txt):
    with open(ass_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    subtitles = []
    in_events = False

    for line in lines:
        # Detecta el inicio de la sección [Events]
        if line.strip().lower() == "[events]":
            in_events = True
            continue

        if in_events:
            # Extrae solo las líneas de subtítulos (que empiezan con "Dialogue:")
            if line.startswith("Dialogue:"):
                # Divide la línea en columnas usando la coma como separador
                parts = line.split(",", 9)  # Separa en máximo 10 partes
                if len(parts) > 9:
                    start_time = parts[1].strip()  # Tiempo de inicio
                    end_time = parts[2].strip()    # Tiempo de fin
                    subtitle_text = parts[9].strip()  # El texto del subtítulo
                    # Formatea la salida incluyendo las marcas de tiempo
                    subtitles.append(f"{start_time},{end_time},{subtitle_text}")

    # Guarda los subtítulos en un archivo .txt
    with open(output_txt, 'w', encoding='utf-8') as out_file:
        out_file.write("\n".join(subtitles))


In [71]:
def process_folder(folder_path, output_folder):
    # Crea la carpeta de salida si no existe
    os.makedirs(output_folder, exist_ok=True)

    # Itera sobre todos los archivos .ass en el folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".ass"):
            ass_path = os.path.join(folder_path, file_name)
            txt_path = os.path.join(output_folder, file_name.replace(".ass", ".txt"))

            print(f"Procesando: {file_name} -> {txt_path}")
            extract_subtitles_with_timestamps(ass_path, txt_path)

In [72]:
output_folder = "OUT_JujutsuKaisen01"
process_folder("JujutsuKaisen01", output_folder)

Procesando: [Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].3.eng.ass -> OUT_JujutsuKaisen01/[Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].3.eng.txt
Procesando: [Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].9.spa.ass -> OUT_JujutsuKaisen01/[Erai-raws] Jujutsu Kaisen - 01 [1080p][Multiple Subtitle].9.spa.txt


In [73]:
def limpiar_texto(texto):
    """Elimina signos de puntuación y caracteres no alfabéticos."""
    # texto = re.sub(r'[^a-zA-Z\s¿?¡!]', '', texto)
    texto = re.sub(r'\{.*?\}', '', texto)
    return texto.lower().strip()

In [74]:
def expand_contractions(text):
    return contractions.fix(text)

In [75]:
# Realizar limpieza de los archivos ya unificados

os.makedirs(output_folder, exist_ok=True)

output_folder_final = "CLEAN_OUT_JujutsuKaisen01"
os.makedirs(output_folder_final, exist_ok=True)

for file_name in os.listdir(output_folder):
    with open(output_folder + "/" + file_name, 'r', encoding='utf-8') as file:
        content = file.read().replace("\\N", " ")
        content_expandido = expand_contractions(content)
        texto_limpio = limpiar_texto(content_expandido)
        file.close()

        with open(output_folder_final + "/" + file_name, 'w', encoding='utf-8') as file:
            file.write(texto_limpio)    

        